# 03 LILAC SH5 Inference: Vader5 2DoF + Manual Hand

Evaluates a trained LILAC mapping with the Vader5 left stick as the 2-DoF arm latent input. The hand stays outside the LILAC model and is controlled manually with the preset X/Y buttons.

In [ ]:
%run ../../../ri_motion_v5_package/init_scripts/init_ipython_setup.py
%run ../../../ri_motion_v5_package/init_scripts/init_qt.py
%run ../package/init_project.py

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.vader5 import *


In [ ]:
# Inference tuning. Model/data paths come from package/paths.py.
OBJECT_SITE_NAMES = DEFAULT_TASK_OBJECT_SITE_NAMES
VADER5_DEADZONE = 0.15
LILAC_ACTION_POS_SCALE = 0.01
LILAC_ACTION_ROT_SCALE = 0.05
RESET_OBJECT_BODY_NAMES = list(DEFAULT_RESET_OBJECT_BODY_NAMES)

# Paper-faithful LILAC: only Vader5 left stick affects the learned arm action.
# The right hand is not part of the model state/action; X/Y only changes the manual preset grasp.
HAND_PRESET_ENABLED = True
HAND_GRIP_SPEED = 2.0
AUTO_GRASP = False
AUTO_GRASP_DISTANCE = 0.075
AUTO_GRASP_SPEED = 2.0


In [ ]:
# Scene
env = MuJoCoParser(rel_xml_path=SCENE_XML, verbose=False)


In [ ]:
# Load the single trained LILAC model. The language stack starts empty; send the first instruction from runtime_language_cli.py.
runtime = load_lilac_inference_runtime(
    action_pos_scale = LILAC_ACTION_POS_SCALE,
    action_rot_scale = LILAC_ACTION_ROT_SCALE,
)
controller = runtime.controller
command_source = runtime.command_source
model = runtime.model
model_config = runtime.model_config
language_index = runtime.language_index
language_dataset = runtime.language_dataset
runtime_command_path = runtime.command_path

print("[LILAC inference] runtime language terminal:")
print("  /opt/anaconda3/envs/ri_motion_v5_env/bin/python project/LILAC_project/scripts/runtime_language_cli.py")


In [ ]:
# Reset environment and initialize viewer
env.reset()
env.init_viewer(
    title     = "LILAC SH5 inference - 2DoF arm + manual hand",
    x_offset  = 0.22,
    width     = 1.0,
    height    = 1.0,
    fontscale = 200,
)
env.viewer.set_cam_info(-54.03, 1.35, -34.22, np.array([ 0.4 , -0.16,  1.12]))
env.viewer.set_transparency(transparent=False)
env.viewer.set_geomgroup(group_2=True, group_3=False)
env.viewer.set_sitegroup(group_0=False)
env.set_p("base_link", "body", (0, 0, 0.01))
env.forward(q=[-0.1], joint_names=["lift_joint"])
env.forward(
    q=[1.12, -0.14, -0.15, -2.48, -0.18, -0.22, -0.02],
    joint_names=[
        "arm_r_joint1", "arm_r_joint2", "arm_r_joint3", "arm_r_joint4",
        "arm_r_joint5", "arm_r_joint6", "arm_r_joint7",
    ],
)
env.forward(
    q=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    joint_names=[
        "arm_l_joint1", "arm_l_joint2", "arm_l_joint3", "arm_l_joint4",
        "arm_l_joint5", "arm_l_joint6", "arm_l_joint7",
    ],
)

reset_object_body_names = [name for name in RESET_OBJECT_BODY_NAMES if name in env.body_names]

def reset_task_objects():
    reset_freejoint_bodies_to_xml_pose(env, reset_object_body_names)

reset_task_objects()
object_state = get_task_object_state(env, site_names=OBJECT_SITE_NAMES)
vader5 = build_vader5_handler(joystick_idx=0, verbose=True)
js = vader5.js

use_r_joint_names = get_right_arm_joint_names(env)
right_hand_joint_names = get_right_finger_joint_names()
ik_solver = build_right_arm_ik_solver(env, use_r_joint_names=use_r_joint_names)
T_rpalm_init = get_right_palm_T(env).copy()
T_rpalm_trgt = T_rpalm_init.copy()
right_grasp_init = 0.0
right_grasp = right_grasp_init
q_arm_cmd = env.get_qpos(joint_names=use_r_joint_names).copy()
q_hand_cmd = get_right_finger_qpos(right_grasp)
env.forward(q=q_arm_cmd, joint_names=use_r_joint_names)
env.forward(q=q_hand_cmd, joint_names=right_hand_joint_names)
env.set_zero_qvel()
state = vader5.get_state(update_first=False)
z_raw = np.zeros(2)
z = np.zeros(2)
lilac_info = {
    "source": "model",
    "utterance": controller.active_utterance(),
    "alpha": 1.0,
    "action": np.zeros(6),
}


In [ ]:
tmr_control = SimpleTimer(name="LILAC 2DoF", Hz=CONTROL_HZ, verbose=True)
tmr_render = SimpleTimer(name="Render", Hz=30, verbose=True)
tmr_language = SimpleTimer(name="LanguageCommand", Hz=5, verbose=False)
tmr_control.start()
tmr_render.start()
tmr_language.start()
env.reset_wall_time()

prev_button_a = 0
prev_button_b = 0

try:
    while env.is_viewer_alive():
        env.increase_wall_time()

        if tmr_language.do_run():
            events = command_source.poll(controller)
            for event in events:
                print("[language]", event)
            tmr_language.end()

        if tmr_control.do_run():
            state = vader5.get_state(update_first=True)
            object_state = get_task_object_state(env, site_names=OBJECT_SITE_NAMES)

            # A resets the arm target and manual hand preset. X/Y controls only the hand preset.
            button_a = int(js.get_button(VADER5_BUTTON_A))
            button_b = int(js.get_button(VADER5_BUTTON_B))
            a_pressed = button_a == 1 and prev_button_a == 0
            b_pressed = button_b == 1 and prev_button_b == 0
            prev_button_a = button_a
            prev_button_b = button_b
            if a_pressed:
                T_rpalm_trgt = T_rpalm_init.copy()
                right_grasp = right_grasp_init
                reset_task_objects()
                object_state = get_task_object_state(env, site_names=OBJECT_SITE_NAMES)
                vader5.reset_motion_memory(rot_dir="yaw")
                print("[home] returned to initial right-palm target, hand preset, and task objects")

            if b_pressed:
                popped = controller.pop_correction()
                active_text = controller.active_utterance() or "<none>"
                print("[language] B pop:", popped, "->", active_text)

            q_arm_curr = env.get_qpos(joint_names=use_r_joint_names)
            ee_pose_curr = T_to_pose6(get_right_palm_T(env))
            state_vec = np.concatenate([q_arm_curr, ee_pose_curr, object_state])

            z_raw = vader5_state_to_latent_z(state, th=VADER5_DEADZONE)
            z = z_raw
            T_rpalm_trgt, lilac_info = controller.safe_update_target(
                T_curr = T_rpalm_trgt,
                state  = state_vec,
                z      = z,
            )
            lilac_info["z_raw"] = z_raw.copy()

            if AUTO_GRASP:
                p_hand = T_to_pose6(T_rpalm_trgt)[:3]
                dist = np.linalg.norm(p_hand - object_state[:3])
                if dist < AUTO_GRASP_DISTANCE:
                    right_grasp = float(
                        np.clip(
                            right_grasp + AUTO_GRASP_SPEED * tmr_control.dt,
                            0.0,
                            1.0,
                        )
                    )

            if HAND_PRESET_ENABLED:
                right_grasp = update_right_finger_command(
                    vader5      = vader5,
                    right_grasp = right_grasp,
                    dt          = tmr_control.dt,
                    grip_speed  = HAND_GRIP_SPEED,
                )

            tmr_control.end()

        q_r_best, ik_info = solve_right_palm_ik(
            env               = env,
            ik_solver         = ik_solver,
            T_rpalm_trgt      = T_rpalm_trgt,
            use_r_joint_names = use_r_joint_names,
        )
        qpos_used_all, joint_names_all = forward_right_arm_and_hand(
            env               = env,
            q_arm             = q_r_best,
            right_grasp       = right_grasp,
            use_r_joint_names = use_r_joint_names,
        )

        if tmr_render.do_run():
            active = str(lilac_info.get("utterance", controller.active_utterance()))[:42]
            env.plot_T(T=T_rpalm_trgt, axis_len=0.12, axis_width=0.006, label="target")
            env.viewer_text_overlay("Mode", "LILAC 2DoF arm + manual hand", loc="top left")
            env.viewer_text_overlay("Control", "left stick:arm A:home B:pop X/Y:hand", loc="top left")
            action = np.asarray(lilac_info.get("action", np.zeros(6)), dtype=np.float64)
            env.viewer_text_overlay("z", "[%+.2f %+.2f]" % (z[0], z[1]), loc="top left")
            env.viewer_text_overlay("dxyz", "[%+.3f %+.3f %+.3f]" % (action[0], action[1], action[2]), loc="top left")
            drpy_deg = action[3:] * 180.0 / np.pi
            env.viewer_text_overlay("drpy", "[%+.2f %+.2f %+.2f] deg" % (drpy_deg[0], drpy_deg[1], drpy_deg[2]), loc="top left")
            env.viewer_text_overlay("Hand", "preset %.2f" % right_grasp, loc="top left")
            env.render()
            tmr_render.end()
except KeyboardInterrupt:
    print("Interrupted.")
finally:
    env.close_viewer()
    vader5.close()
    imshow(env.final_rgb_img, title="Final scene", title_fs=8)
